# CODY-SAM3 — Phase 3 - External validation - v2 (EXPLORATION of all calibration sets)

Same calibration objective as v1 (**CODY-2**: `Jaccard - 0.35*fp_excess - 0.35*fn_key`), but instead of a single manually chosen calibration set, **all** possible calibration sets of size *k* are tested (C(n,k), sampled if too many) and we report:

1. the **distribution** of the metrics over all sets (the robust/publishable result);
2. the **top 5** sets by overall metric (Jaccard, Hamming), main & restricted;
3. the **top 5** sets by confusion counter (TP, TN, FP, FN).

For each set: **held-out** (external validation) and **all-patients** views, under the main present/absent and restricted definitions (>=3/5, >=4/5, 5/5).

> **Methodological note.** The **distribution** is the result to report (robustness to the choice of calibration set). The **top 5** lists are **diagnostic** (understanding which patients are representative); reporting them as performance would amount to selecting on the test data. The paper reports the clinician's a priori set (see v1), as in CODY-2.

**GPU**: not required (post-processing of window-level probabilities).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q openpyxl pandas numpy matplotlib
import pandas as pd, numpy as np, gzip, re, math, time
from pathlib import Path
from itertools import combinations

## 2. Configuration

In [ ]:
DRIVE   = Path('/content/drive/MyDrive/sam_3_infer')
RESULTS = DRIVE / 'external_validation'
GT_XLSX = DRIVE / 'dataset_inference.xlsx'

DATASETS = ['dataset_2','dataset_3']     # dataset_1 trop petit (illustratif)
K        = {'dataset_1':1,'dataset_2':5,'dataset_3':5}   # taille set de calibration
SHEETS   = {'dataset_1':'dataset_1','dataset_2':'dataset_2','dataset_3':'dataset_3'}

# Objectif de calibration CODY-2 (identique a calibrate_persite.py)
AGG_GRID    = ['p70','p90','p95','max']
OBJECTIVE   = 'jaccard'
EXTRA_PEN   = 0.35
FN_PEN      = 0.35
FN_KEY      = ['Dystonia','Myoclonus','Chorea']
CALIB_FAMILY= 'main'    # definition utilisee pour CALIBRER
CALIB_LEVEL = 3

MAX_SETS = 2000         # if C(n,k) exceeds this, sample (reproducible)
SEED     = 0
PHEN=['Dystonia','Tremor','Myoclonus','Chorea','Athetosis','Ballismus','Stereotypies','Tics']
THR_GRID=[0.08,0.10,0.12,0.15,0.18,0.20,0.22,0.25,0.28,0.30,0.33,0.35,0.38,0.40,0.45,0.50,0.60,0.70,0.80,0.90,0.95,0.99]
MIN_THR={'Tremor':0.35,'Tics':0.35,'Ballismus':0.30,'Stereotypies':0.25,'Athetosis':0.20}
MAX_THR={'Dystonia':0.55,'Myoclonus':0.50,'Chorea':0.55}
LABEL_DEFS=[('main',3),('main',4),('main',5),('restricted',3),('restricted',4),('restricted',5)]

## 3. Fonctions (objectif CODY-2)

In [ ]:
def agg_probs(p,m):
    p=p[np.isfinite(p)]
    if p.size==0: return np.nan
    if m=='max': return float(p.max())
    if m=='mean': return float(p.mean())
    if m=='median': return float(np.median(p))
    if m=='noisy_or': return float(1-np.prod(1-p))
    mm=re.match(r'p(\d{2,3})',m); return float(np.percentile(p,float(mm.group(1))))

def parse_votes(xlsx,sheet):
    raw=pd.read_excel(xlsx,sheet_name=sheet,header=None)
    rr=raw.iloc[0].tolist(); pr=raw.iloc[1].tolist()
    starts=[j for j,v in enumerate(rr) if isinstance(v,str) and v.strip() and v.strip().lower()!='nan']
    data=raw.iloc[2:].reset_index(drop=True); pids=data.iloc[:,1].astype(str).str.strip()
    rec={}
    for i,pid in enumerate(pids):
        if pid in ('','nan','None'): continue
        rec[pid]={}
        for ph in PHEN:
            vals=[]
            for s in starts:
                for off in range(8):
                    c=s+off
                    if c<raw.shape[1] and str(pr[c]).strip()==ph:
                        try: vals.append(float(str(data.iloc[i,c]).replace(',','.')))
                        except: pass
            rec[pid][ph]=[v for v in vals if v in (0.,1.)]
    return rec

def label_of(votes,family,level):
    if not votes: return None
    npos=sum(v==1 for v in votes); present=npos>=level
    if family=='main': return 1 if present else 0
    if present: return 1
    if npos==0: return 0
    return None

def jaccard(Yt,Yp):
    inter=np.sum(Yt&Yp,axis=1); union=np.sum(Yt|Yp,axis=1)
    return float(np.mean(np.where(union==0,1.0,inter/union)))
def exact_match(Yt,Yp): return float(np.mean(np.all(Yt==Yp,axis=1)))
def macro_f1(Yt,Yp):
    f=[]
    for j in range(Yt.shape[1]):
        tp=int(((Yt[:,j]==1)&(Yp[:,j]==1)).sum());fp=int(((Yt[:,j]==0)&(Yp[:,j]==1)).sum());fn=int(((Yt[:,j]==1)&(Yp[:,j]==0)).sum())
        f.append(1.0 if (tp==0 and fp==0 and fn==0) else (2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0.0))
    return float(np.mean(f))
def fp_excess_norm(Yt,Yp):
    return float(np.mean(np.maximum(0,Yp.sum(1)-Yt.sum(1)))/max(1.0,Yt.shape[1]))
def fn_key_rate(Yt,Yp,key):
    idx=[PHEN.index(l) for l in key if l in PHEN]; vals=[]
    for j in idx:
        pos=int((Yt[:,j]==1).sum())
        if pos==0: continue
        vals.append(int(((Yt[:,j]==1)&(Yp[:,j]==0)).sum())/max(1,pos))
    return float(np.mean(vals)) if vals else 0.0
OBJ={'jaccard':jaccard,'exact_match':exact_match,'macro_f1':macro_f1}

def thr_candidates(l):
    floor=MIN_THR.get(l,min(THR_GRID)); ceil=MAX_THR.get(l,max(THR_GRID))
    v=[t for t in THR_GRID if floor-1e-9<=t<=ceil+1e-9]
    return v if v else [min(max(THR_GRID[0],floor),ceil)]

In [ ]:
def calibrate(calib, rec, Pcache, family, level, max_iter=16):
    obj=OBJ[OBJECTIVE]
    Yt=np.array([[ (0 if label_of(rec[p][ph],family,level) is None else label_of(rec[p][ph],family,level)) for ph in PHEN] for p in calib],dtype=int)
    def score(asel,tsel):
        P=np.array([[Pcache[(PHEN[j],asel[j])][p] for j in range(len(PHEN))] for p in calib])
        Yp=(P>=np.array(tsel)[None,:]).astype(int)
        return obj(Yt,Yp)-EXTRA_PEN*fp_excess_norm(Yt,Yp)-FN_PEN*fn_key_rate(Yt,Yp,FN_KEY)
    asel=['p95']*len(PHEN); tsel=[thr_candidates(l)[len(thr_candidates(l))//2] for l in PHEN]
    best=score(asel,tsel)
    for _ in range(max_iter):
        improved=False
        for j,l in enumerate(PHEN):
            for m in AGG_GRID:
                for t in thr_candidates(l):
                    ca=asel.copy();ct=tsel.copy();ca[j]=m;ct[j]=t
                    sc=score(ca,ct)
                    if sc>best+1e-12: best,asel,tsel=sc,ca,ct;improved=True
        if not improved: break
    return {PHEN[j]:(asel[j],float(tsel[j])) for j in range(len(PHEN))}

def evaluate(rules, pats, rec, Pcache, family, level):
    TP=TN=FP=FN=0; ham=[]; jac=[]
    for p in pats:
        corr=tot=inter=union=0
        for ph in PHEN:
            tl=label_of(rec[p][ph],family,level)
            if tl is None: continue
            am,t=rules[ph]; pr=int(Pcache[(ph,am)][p]>=t); tot+=1; corr+=int(pr==tl)
            if pr==1 and tl==1: TP+=1;inter+=1;union+=1
            elif pr==1 and tl==0: FP+=1;union+=1
            elif pr==0 and tl==1: FN+=1;union+=1
            else: TN+=1
        if tot: ham.append(corr/tot)
        jac.append(inter/union if union>0 else 1.0)
    return dict(TP=TP,TN=TN,FP=FP,FN=FN,Hamming=np.mean(ham) if ham else np.nan,Jaccard=np.mean(jac) if jac else np.nan)

## 4. Exploration de tous les sets

In [ ]:
def run_dataset(ds):
    k=K[ds]
    win=RESULTS/ds/'reports'/'tables'/'inference_window_predictions.csv.gz'
    with gzip.open(win,'rt') as f: w=pd.read_csv(f)
    w['patient_id']=w['patient_id'].astype(str).str.strip()
    rec=parse_votes(GT_XLSX,SHEETS[ds])
    patients=sorted([p for p in rec if p in set(w['patient_id'])])
    Pcache={(ph,m):{p:agg_probs(w.loc[w.patient_id==p,f'prob__{ph}'].to_numpy(float),m) for p in patients}
            for ph in PHEN if f'prob__{ph}' in w.columns for m in AGG_GRID}
    total=math.comb(len(patients),k)
    if total<=MAX_SETS:
        sets=list(combinations(patients,k)); mode=f'exhaustif ({total})'
    else:
        rng=np.random.default_rng(SEED); seen=set(); sets=[]
        while len(sets)<MAX_SETS:
            s=tuple(sorted(rng.choice(patients,k,replace=False)))
            if s not in seen: seen.add(s); sets.append(s)
        mode=f'sampled ({MAX_SETS}/{total})'
    print(f'{ds}: {len(patients)} patients, k={k}, sets={mode}')
    t0=time.time(); rows=[]
    for calib in sets:
        calib=list(calib); test=[p for p in patients if p not in set(calib)]
        rules=calibrate(calib,rec,Pcache,CALIB_FAMILY,CALIB_LEVEL)
        for family,level in LABEL_DEFS:
            ho=evaluate(rules,test,rec,Pcache,family,level)
            al=evaluate(rules,patients,rec,Pcache,family,level)
            rows.append(dict(dataset=ds,calib=';'.join(calib),heldout=';'.join(test),
                definition=family,agreement=f'{level}of5',
                ho_Hamming=ho['Hamming'],ho_Jaccard=ho['Jaccard'],ho_TP=ho['TP'],ho_TN=ho['TN'],ho_FP=ho['FP'],ho_FN=ho['FN'],
                all_Hamming=al['Hamming'],all_Jaccard=al['Jaccard'],all_TP=al['TP'],all_TN=al['TN'],all_FP=al['FP'],all_FN=al['FN']))
    print(f'  -> {len(rows)} lignes en {time.time()-t0:.0f}s')
    return pd.DataFrame(rows)

ALL=[]
for ds in DATASETS:
    d=run_dataset(ds)
    out=RESULTS/ds/'eval_allsets'; out.mkdir(parents=True,exist_ok=True)
    d.to_csv(out/f'allsets_{ds}.csv.gz',index=False,compression='gzip')
    ALL.append(d)
BIG=pd.concat(ALL,ignore_index=True)
print('Total:',len(BIG),'lignes')

## 5. Distributions (the publishable result)

In [ ]:
rows=[]
for (ds,dfam,agr),g in BIG.groupby(['dataset','definition','agreement']):
    for metric in ['ho_Jaccard','ho_Hamming']:
        v=g[metric].dropna().to_numpy()
        if v.size==0: continue
        rows.append(dict(dataset=ds,definition=dfam,agreement=agr,metric=metric.replace('ho_',''),
            median=np.median(v),q05=np.percentile(v,5),q95=np.percentile(v,95),n_sets=g['calib'].nunique()))
dist=pd.DataFrame(rows)
display(dist.round(3))
dist.to_csv(RESULTS/'SUMMARY_allsets_distribution.csv',index=False)

## 6. Top 5 by overall metric (Jaccard, Hamming) — diagnostic

In [ ]:
def top5_metric(metric):
    out=[]
    for (ds,dfam,agr),g in BIG.groupby(['dataset','definition','agreement']):
        t=g.sort_values(metric,ascending=False).head(5)
        out.append(t)
    return pd.concat(out,ignore_index=True)

for metric in ['ho_Jaccard','ho_Hamming']:
    t=top5_metric(metric)
    cols=['dataset','definition','agreement','calib','ho_Hamming','ho_Jaccard','ho_TP','ho_TN','ho_FP','ho_FN','all_Hamming','all_Jaccard']
    t[cols].to_csv(RESULTS/f'TOP5_{metric}.csv',index=False)
print('Exemple — Top5 Jaccard held-out, main 3/5:')
ex=top5_metric('ho_Jaccard')
display(ex[(ex.definition=='main')&(ex.agreement=='3of5')][['dataset','calib','ho_Hamming','ho_Jaccard','ho_TP','ho_TN','ho_FP','ho_FN','all_Jaccard']].round(3))

## 7. Top 5 par compteur de confusion (TP, TN, FP, FN) — diagnostique

In [ ]:
def top5_conf(counter,ascending):
    out=[]
    for (ds,dfam,agr),g in BIG.groupby(['dataset','definition','agreement']):
        out.append(g.sort_values(counter,ascending=ascending).head(5))
    return pd.concat(out,ignore_index=True)

for counter,asc in [('ho_TP',False),('ho_TN',False),('ho_FP',True),('ho_FN',True)]:
    t=top5_conf(counter,asc)
    cols=['dataset','definition','agreement','calib','ho_TP','ho_TN','ho_FP','ho_FN','ho_Hamming','ho_Jaccard']
    t[cols].to_csv(RESULTS/f'TOP5_{counter}.csv',index=False)
print('Exemple — Top5 max TP held-out, main 3/5:')
ex=top5_conf('ho_TP',False)
display(ex[(ex.definition=='main')&(ex.agreement=='3of5')][['dataset','calib','ho_TP','ho_FP','ho_FN','ho_TN','ho_Jaccard']])

## 8. Saved outputs

Under `external_validation/`: `eval_allsets/allsets_<ds>.csv.gz` (full table), `SUMMARY_allsets_distribution.csv`, `TOP5_*.csv`.

**Reminder**: for the paper, report the clinician's a priori set (v1 notebook). These tops are for understanding the selection, not for reporting performance.